In [ ]:
import os
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
import warnings

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [ ]:
def train_linear(df, y_pred_baseline):
  df = df.dropna()
  df["datetime_idx"] = pd.to_datetime(df["datetime_idx"].astype(str))

  valid_test_dates = [str(date) for date in set(df["datetime_idx"]).intersection(set(y_pred_baseline["datetime_idx"]))]

  df.set_index("datetime_idx", inplace=True)
  y_pred_baseline.set_index("datetime_idx", inplace=True)

  X = df[[
      'minutes_from_open', 'minutes_to_close',
      'wait_time_lag_day', 'wait_time_lag_week', 'wait_time_lag_year',
      'wt_rolling_week_avg', 'wt_rolling_month_avg', 'wt_rolling_year_avg',
      'wt_rolling_week_max', 'wt_rolling_month_max', 'wt_rolling_year_max',
      'wt_rolling_week_std', 'wt_rolling_month_std', 'wt_rolling_year_std',
      'duration', 'min_height', 'has_express', "has_ea",
      'is_kids_ride', 'is_thrill_ride', 'is_dark_ride', 'is_water_ride', 'has_indoor_queue',
      'temp', 'rhum', 'prcp', 'wspd', 'pres', 'cldc', 'coco',
      'is_holiday', 'is_major_holiday', 'is_annual_event',
      'year', 'month_sin', 'month_cos', 'day_sin', 'day_cos', 'time_sin', 'time_cos',
      'day_of_week_sin', 'day_of_week_cos'
  ]]
  y = df["wait_time"]

  train_split = str(df[df.index < pd.to_datetime("2025-01-01 00:0:00")].index.max())

  X_train = X.loc[:train_split]
  X_test = X.loc[train_split:]
  X_test = X_test.loc[valid_test_dates].sort_index()

  X_train_index = X_train.index
  X_test_index = X_test.index

  scaler = StandardScaler()
  X_train = pd.DataFrame(scaler.fit_transform(X_train), index=X_train_index)
  X_test = pd.DataFrame(scaler.fit_transform(X_test), index=X_test_index)

  y_train = y.loc[:train_split]
  y_test = y.loc[train_split:]
  y_test = y_test.loc[valid_test_dates].sort_index()
  y_pred_baseline = y_pred_baseline.loc[valid_test_dates].sort_index()

  linear_model = LinearRegression()

  linear_model.fit(X_train, y_train)

  y_pred = pd.DataFrame(linear_model.predict(X_test)).set_index(y_test.index)

  baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
  baseline_rmse = root_mean_squared_error(y_test, y_pred_baseline)
  baseline_wape = sum(np.abs(np.array(y_test) - np.array(y_pred_baseline["wait_time"]))) / sum(y_test)

  mae = mean_absolute_error(y_test, y_pred)
  rmse = root_mean_squared_error(y_test, y_pred)
  wape = sum(np.abs(np.array(y_test) - np.array(y_pred[0]))) / sum(y_test)

  return linear_model, baseline_mae, baseline_rmse, baseline_wape, mae, rmse, wape, y_pred

def theme_park_linear(folder_path):
  for file_name in os.listdir(f"{folder_path}Wait_Time_Features/"):
    ride_name = file_name.replace("_features.parquet", "")
    try:
      baseline_predictions = pd.read_parquet(f"{folder_path}Baseline_Predictions/{ride_name}.parquet")
      features = pd.read_parquet(f"{folder_path}Wait_Time_Features/{file_name}")
      results = pd.read_csv(f"{folder_path}Results/linear_results.csv")

      linear_model, baseline_mae, baseline_rmse, baseline_wape, mae, rmse, wape, predictions = train_linear(features, baseline_predictions)

      results = pd.concat([results, pd.DataFrame({"ride_name": ride_name, "baseline_mae": baseline_mae, "baseline_rmse": baseline_rmse, "baseline_wape": baseline_wape, "mae": mae, "rmse": rmse, "wape": wape}, index=[0])], ignore_index=True)

      results.to_csv(f"{folder_path}Results/linear_results.csv", index=False)
      predictions.to_csv(f"{folder_path}Predictions/Linear/{ride_name}_predictions.csv", index=False)
    except Exception as e:
      continue


In [ ]:
# Create Linear Models for Islands of Adventure
theme_park_linear("/content/drive/MyDrive/Theme_Park_Data/Universal_IOA/")

# Create Linear Models for Universal Studios
theme_park_linear("/content/drive/MyDrive/Theme_Park_Data/Universal_Studios_FL/")